# Dérivée par rapport à des tenseurs
Pas de code à trous dans la cellule suivante

In [ ]:
from __future__ import annotations

from collections import defaultdict
from collections.abc import Callable
from math import exp

import numpy as np


class Op:
    def eval(self):
        raise NotImplementedError("Subclasses should implement this method.")

    def diff(self, wrt: Variable, direction: Variable) -> Op:
        raise NotImplementedError("Subclasses should implement this method.")

    def __add__(self, other: "Op"):
        return Add(self, other)

    def __mul__(self, other: "Op"):
        return Mul(self, other)

    def __matmul__(self, other: "Op"):
        return MatMul(self, other)

    def __sub__(self, other: "Op"):
        return Sub(self, other)


class ConstantZero(Op):
    def __init__(self):
        pass

    def __repr__(self):
        return "0"

    def eval(self):
        return 0.0


class Variable(Op):
    def __init__(self, name: str, value: np.ndarray):
        self.name = name
        self.value = value

    def __repr__(self):
        return self.name

    def eval(self, perturbed_variable=None):
        return self.value

    def diff(self, wrt: Variable, direction: Variable) -> Op:
        if self == wrt:
            return direction
        else:
            return ConstantZero()


class Add(Op):
    def __init__(self, left: Op, right: Op):
        self.left = left
        self.right = right

    def __repr__(self):
        return f"({self.left} + {self.right})"

    def diff(self, wrt: Variable, direction: Variable) -> Op:
        left_diff = self.left.diff(wrt, direction)
        right_diff = self.right.diff(wrt, direction)

        if isinstance(left_diff, ConstantZero) and isinstance(right_diff, ConstantZero):
            return ConstantZero()
        if isinstance(left_diff, ConstantZero):
            return right_diff
        if isinstance(right_diff, ConstantZero):
            return left_diff

        return Add(left_diff, right_diff)

    def eval(self):
        return self.left.eval() + self.right.eval()


class Sub(Op):
    def __init__(self, left: Op, right: Op):
        self.left = left
        self.right = right

    def __repr__(self):
        return f"({self.left} - {self.right})"

    def diff(self, wrt: Variable, direction: Variable) -> Op:
        return Sub(self.left.diff(wrt, direction), self.right.diff(wrt, direction))

    def eval(self):
        return self.left.eval() - self.right.eval()


class Linear(Op):
    """Offre la possibilité de définir la dérivée sur toute un classe de fonctions"""
    def __init__(self, arg: Op, repr: str, linear_operation: Callable):
        self.operation = linear_operation
        self.arg = arg
        self.repr = repr

    def __repr__(self):
        return f"{self.repr}({self.arg})"

    def eval(self):
        return self.operation(self.arg.eval())

    def diff(self, wrt: Variable, direction: Variable) -> Op:
        arg_diff = self.arg.diff(wrt, direction)

        # if isinstance(arg_diff, ConstantZero):
        #     return ConstantZero()

        # else
        return Linear(
            repr=self.repr,
            arg=arg_diff,
            linear_operation=self.operation,
        )


class Bilinear(Op):
    """Offre la possibilité de définir la dérivée sur toute un classe de fonctions"""
    def __init__(self, left: Op, right: Op, repr: Callable, operation: Callable):
        self.operation = operation
        self.repr = repr
        self.left = left
        self.right = right

    def __repr__(self):
        return self.repr(self.left, self.right)

    def eval(self):
        left_eval = self.left.eval()
        right_eval = self.right.eval()

        if isinstance(left_eval, float) or isinstance(right_eval, float):
            return left_eval * right_eval

        return self.operation(left_eval, right_eval)

    def diff(self, wrt: Variable, direction: Variable) -> Op:
        left_diff = self.left.diff(wrt, direction)
        right_diff = self.right.diff(wrt, direction)

        ## Simplification rules (optional)
        # if isinstance(left_diff, ConstantZero) and isinstance(right_diff, ConstantZero):
        #     return ConstantZero()

        # if isinstance(left_diff, ConstantZero):
        #     return Bilinear(
        #         repr=self.repr,
        #         left=self.left,
        #         right=right_diff,
        #         operation=self.operation,
        #     )
        # if isinstance(right_diff, ConstantZero):
        #     return Bilinear(
        #         repr=self.repr,
        #         left=left_diff,
        #         right=self.right,
        #         operation=self.operation,
        #     )

        return Add(
            Bilinear(
                repr=self.repr,
                left=left_diff,
                right=self.right,
                operation=self.operation,
            ),
            Bilinear(
                repr=self.repr,
                left=self.left,
                right=right_diff,
                operation=self.operation,
            ),
        )


class Mul(Bilinear):
    def __init__(self, left: Op, right: Op):
        super().__init__(
            repr=lambda l, r: f"({l} . {r})",
            left=left,
            right=right,
            operation=lambda l, r: l * r,
        )


class MatMul(Bilinear):
    def __init__(self, left: Op, right: Op):
        def repr(a, b):
            return f"({a} @ {b})"

        super().__init__(
            repr=repr,
            left=left,
            right=right,
            operation=lambda l, r: l @ r,
        )


def square(x: Op) -> Op:
    return Mul(x, x)




    

In [5]:
# Example usage

a = Variable(name="a", value=np.random.randn(3, 2))
b = Variable(name="b", value=np.random.randn(2, 4))

expr = Bilinear(
    a, b, repr=lambda l, r: f"B({l}, {r})", operation=lambda l, r: l @ r
)

print(
    expr.diff(
        wrt=a,
        direction=Variable(name="delta_a", value=np.ones((3, 2))),
    )
)

print(expr.eval())


(B(delta_a, b) + B(a, 0))
[[ 2.69804818e+00  8.81910047e-01 -1.49812147e+00  1.83268799e-01]
 [ 4.02272307e-01 -1.46075703e-01 -1.16119603e+00 -8.05634566e-01]
 [ 3.70370299e-01  1.13363795e-01 -2.31665370e-01  2.05366957e-03]]


# Exercice 03a
Même exercice que Exercice 02b, mais en maniant des tenseurs plutôt que des variables individuelles.

Objectif de l'algo : estimer `a_true`, `b_true`, `c_true`, qui sont néanmoins des variables réelles et non pas des vecteurs/matrice/tenseurs.

Objectif de l'exercice, pour les participants : Observer la différence de complexité de la dérivée en comparaison de ce que l'on avait dans l'exercice 02b.


## Préparation des données et du modèle

In [8]:
import numpy as np


# generate data
nb_samples = 100

a_true = 3.0
b_true = -5.0
c_true = 10.0

X_obs = np.random.randn(nb_samples, 1)

# linear relation with X_obs plus some noise
Y_obs = (
    a_true * X_obs * X_obs
    + b_true * X_obs
    + c_true
    + 0.1 * np.random.randn(nb_samples, 1)
)


def model(
    a_est: Op,
    b_est: Op,
    c_est: Op,
    x_obs: Op,
) -> Op:
    y_pred = a_est * x_obs * x_obs + b_est * x_obs + c_est
    return y_pred


# init "constants"
x_obs = Variable(name="x_obs", value=X_obs)
y_obs = Variable(name="y_obs", value=Y_obs)

# init variables to optimize
a_est = Variable(name="a_est", value=np.array([0.0]))
b_est = Variable(name="b_est", value=np.array([0.0]))
c_est = Variable(name="c_est", value=np.array([0.0]))



## Gradient descent

In [9]:
for iteration in range(20):
    y_pred = model(a_est, b_est, c_est, x_obs)

    loss = Linear(
        arg=square((y_pred - y_obs)),
        repr="mean",
        linear_operation=lambda x: np.mean(x),
    )

    print(
        "Gradient a:",
        loss.diff(wrt=a_est, direction=Variable(name="one", value=np.array([1.0]))),
    )

    print(
        "Gradient b:",
        loss.diff(wrt=b_est, direction=Variable(name="one", value=np.array([1.0]))),
    )

    break

    delta_a = loss.diff(
        wrt=a_est, direction=Variable(name="one", value=np.array([1.0]))
    ).eval()
    delta_b = loss.diff(
        wrt=b_est, direction=Variable(name="one", value=np.array([1.0]))
    ).eval()
    learning_rate = 1e-1

    a_est.value -= learning_rate * delta_a
    b_est.value -= learning_rate * delta_b

    print("loss = ", loss.eval(), "\ta_est = ", a_est.value, "\tb_est = ", b_est.value)


Gradient a: mean(((((((((one . x_obs) + (a_est . 0)) . x_obs) + ((a_est . x_obs) . 0)) + ((0 . x_obs) + (b_est . 0))) - 0) . (((((a_est . x_obs) . x_obs) + (b_est . x_obs)) + c_est) - y_obs)) + ((((((a_est . x_obs) . x_obs) + (b_est . x_obs)) + c_est) - y_obs) . ((((((one . x_obs) + (a_est . 0)) . x_obs) + ((a_est . x_obs) . 0)) + ((0 . x_obs) + (b_est . 0))) - 0))))
Gradient b: mean(((((((((0 . x_obs) + (a_est . 0)) . x_obs) + ((a_est . x_obs) . 0)) + ((one . x_obs) + (b_est . 0))) - 0) . (((((a_est . x_obs) . x_obs) + (b_est . x_obs)) + c_est) - y_obs)) + ((((((a_est . x_obs) . x_obs) + (b_est . x_obs)) + c_est) - y_obs) . ((((((0 . x_obs) + (a_est . 0)) . x_obs) + ((a_est . x_obs) . 0)) + ((one . x_obs) + (b_est . 0))) - 0))))
